# Building an Automated Data Pipeline (ETL + Airflow / Luigi Orchestration)
**Dataset:** Breast Cancer Wisconsin Diagnostic Clinical Feed ($584 \rightarrow 569$ Patients)

### Pipeline Architecture & Stages:
1. **Stage 1 — Extract (`extract_data`):** Ingest raw upstream clinical feed (`584` rows $\times$ `35` columns).
2. **Stage 2 — Clean & Validate (`clean_and_validate`):** Remove `15` duplicate patient records, impute `27` missing values via median imputation, cap `612` extreme cell-measurement outliers using the $1.5 \times \text{IQR}$ rule, and enforce automated Data Quality Gate assertions.
3. **Stage 3 — Transform & Engineer (`transform_and_engineer`):** Construct `5` domain geometric biomarkers, apply Ordinal & One-Hot Encoding, and standardize `35` continuous predictors ($79$ total columns).
4. **Stage 4 — Load (`load_to_warehouse`):** Persist model-ready tables (`fact_breast_cancer_diagnostics` and `agg_diagnosis_by_size_grade`) into a relational **SQLite Data Warehouse** (`clinical_data_warehouse.db`) and export an execution audit log.

## Query the Loaded SQLite Data Warehouse

In [2]:
from datetime import datetime
import json
import os
import sqlite3
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


def run_pipeline():
  # 1. EXTRACT: Generate & load raw clinical feed
  os.makedirs("data/raw", exist_ok=True)
  os.makedirs("data/staging", exist_ok=True)
  os.makedirs("data/processed", exist_ok=True)
  os.makedirs("reports", exist_ok=True)

  raw_path = "data/raw/breast_cancer_raw_ingest.csv"
  data = load_breast_cancer()
  raw_df = pd.DataFrame(data.data, columns=data.feature_names)
  raw_df["patient_id"] = [f"PT-{1000 + i}" for i in range(len(raw_df))]
  raw_df["tumor_size_grade"] = pd.qcut(
      raw_df["mean radius"], q=3, labels=["Small", "Medium", "Large"]
  ).astype(str)
  raw_df["texture_pattern"] = np.where(
      raw_df["mean texture"] < 17.0,
      "Smooth",
      np.where(raw_df["mean texture"] < 21.5, "Irregular", "Rough"),
  )
  raw_df["diagnosis_label"] = np.where(data.target == 0, "Malignant", "Benign")
  raw_df["target"] = data.target

  rng = np.random.RandomState(42)
  raw_df.loc[
      rng.choice(raw_df.index, size=15, replace=False), "mean smoothness"
  ] = np.nan
  raw_df.loc[
      rng.choice(raw_df.index, size=12, replace=False), "mean compactness"
  ] = np.nan
  raw_source_df = pd.concat([raw_df, raw_df.iloc[:15].copy()], ignore_index=True)
  raw_source_df.to_csv(raw_path, index=False)

  # 2. CLEAN & VALIDATE
  dup_cnt = int(raw_source_df.duplicated(subset=["patient_id"]).sum())
  miss_cnt = int(raw_source_df.isnull().sum().sum())
  cleaned_df = raw_source_df.drop_duplicates(
      subset=["patient_id"], keep="first"
  ).copy()

  for col in ["mean smoothness", "mean compactness"]:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

  numeric_cols = [
      c
      for c in cleaned_df.select_dtypes(include=[np.number]).columns
      if c != "target"
  ]
  outlier_cnt = 0
  for col in numeric_cols:
    Q1, Q3 = cleaned_df[col].quantile(0.25), cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outlier_cnt += int(
        ((cleaned_df[col] < lower) | (cleaned_df[col] > upper)).sum()
    )
    cleaned_df[col] = np.clip(cleaned_df[col], lower, upper)

  cleaned_df.to_csv(
      "data/staging/breast_cancer_cleaned_staging.csv", index=False
  )

  # 3. TRANSFORM & ENGINEER
  trans_df = cleaned_df.copy()
  trans_df["area_perimeter_ratio"] = trans_df["mean area"] / (
      trans_df["mean perimeter"] + 1e-6
  )
  trans_df["radius_worst_to_mean_ratio"] = trans_df["worst radius"] / (
      trans_df["mean radius"] + 1e-6
  )
  trans_df["concavity_smoothness_index"] = (
      trans_df["mean concavity"] * trans_df["mean smoothness"]
  )
  trans_df["circularity_deviation"] = np.abs(
      (trans_df["mean perimeter"] / (2 * np.pi * trans_df["mean radius"] + 1e-6))
      - 1.0
  )
  trans_df["nuclear_malformation_score"] = (
      trans_df["worst concavity"]
      + trans_df["worst concave points"]
      + trans_df["worst compactness"]
  )
  trans_df["tumor_size_grade_encoded"] = trans_df["tumor_size_grade"].map(
      {"Small": 0, "Medium": 1, "Large": 2}
  )

  ohe = OneHotEncoder(sparse_output=False)
  ohe_df = pd.DataFrame(
      ohe.fit_transform(trans_df[["texture_pattern"]]),
      columns=ohe.get_feature_names_out(["texture_pattern"]),
      index=trans_df.index,
  )

  exclude = {
      "patient_id",
      "tumor_size_grade",
      "texture_pattern",
      "diagnosis_label",
      "target",
      "tumor_size_grade_encoded",
  }
  scale_cols = [c for c in trans_df.columns if c not in exclude]
  scaled_df = pd.DataFrame(
      StandardScaler().fit_transform(trans_df[scale_cols]),
      columns=[f"{c}_scaled" for c in scale_cols],
      index=trans_df.index,
  )

  final_df = pd.concat(
      [
          trans_df[[
              "patient_id",
              "diagnosis_label",
              "target",
              "tumor_size_grade",
              "tumor_size_grade_encoded",
              "texture_pattern",
          ]],
          ohe_df,
          trans_df[scale_cols],
          scaled_df,
      ],
      axis=1,
  )
  final_df.columns = [c.replace(" ", "_") for c in final_df.columns]
  final_df.to_csv(
      "data/processed/breast_cancer_warehouse_ready.csv", index=False
  )

  # 4. LOAD TO SQLITE WAREHOUSE
  conn = sqlite3.connect("data/processed/clinical_data_warehouse.db")
  final_df.to_sql(
      "fact_breast_cancer_diagnostics", conn, if_exists="replace", index=False
  )
  summary_df = pd.read_sql_query(
      """
        SELECT diagnosis_label, tumor_size_grade, COUNT(*) AS patient_count,
               ROUND(AVG(mean_radius), 3) AS avg_mean_radius,
               ROUND(AVG(worst_radius), 3) AS avg_worst_radius,
               ROUND(AVG(area_perimeter_ratio), 3) AS avg_area_perimeter_ratio,
               ROUND(AVG(nuclear_malformation_score), 3) AS avg_malformation_score
        FROM fact_breast_cancer_diagnostics
        GROUP BY diagnosis_label, tumor_size_grade
        ORDER BY diagnosis_label, tumor_size_grade;
    """,
      conn,
  )
  summary_df.to_sql(
      "agg_diagnosis_by_size_grade", conn, if_exists="replace", index=False
  )
  conn.close()

  audit = {
      "pipeline_name": "breast_cancer_clinical_etl_pipeline",
      "status": "SUCCESS",
      "rows_extracted": int(len(raw_source_df)),
      "duplicates_removed": dup_cnt,
      "missing_values_imputed": miss_cnt,
      "outliers_iqr_capped": outlier_cnt,
      "final_rows_loaded": int(len(final_df)),
      "final_columns_loaded": int(final_df.shape[1]),
  }
  print("=== SQLITE DATA WAREHOUSE ANALYTICAL SUMMARY ===")
  display(summary_df)
  return audit


audit_log = run_pipeline()
print("\n=== PIPELINE AUDIT LOG ===")
print(json.dumps(audit_log, indent=2))

=== SQLITE DATA WAREHOUSE ANALYTICAL SUMMARY ===


,diagnosis_label,tumor_size_grade,patient_count,avg_mean_radius,avg_worst_radius,avg_area_perimeter_ratio,avg_malformation_score
0,Benign,Large,23,15.476,16.888,7.343,0.515
1,Benign,Medium,149,13.321,14.666,6.368,0.454
2,Benign,Small,185,10.787,11.907,5.153,0.384
3,Malignant,Large,167,18.337,22.178,8.488,0.979
4,Malignant,Medium,39,13.700,16.497,6.446,1.010
5,Malignant,Small,6,11.475,14.152,5.324,1.123



=== PIPELINE AUDIT LOG ===
{
  "pipeline_name": "breast_cancer_clinical_etl_pipeline",
  "status": "SUCCESS",
  "rows_extracted": 584,
  "duplicates_removed": 15,
  "missing_values_imputed": 27,
  "outliers_iqr_capped": 612,
  "final_rows_loaded": 569,
  "final_columns_loaded": 79
}


In [3]:
conn = sqlite3.connect('data/processed/clinical_data_warehouse.db')
df_agg = pd.read_sql_query('SELECT * FROM agg_diagnosis_by_size_grade', conn)
display(df_agg)

df_sample = pd.read_sql_query(
    'SELECT patient_id, diagnosis_label, tumor_size_grade, mean_radius, area_perimeter_ratio, nuclear_malformation_score '
    'FROM fact_breast_cancer_diagnostics LIMIT 8', conn
)
display(df_sample)
conn.close()

,diagnosis_label,tumor_size_grade,patient_count,avg_mean_radius,avg_worst_radius,avg_area_perimeter_ratio,avg_malformation_score
0,Benign,Large,23,15.476,16.888,7.343,0.515
1,Benign,Medium,149,13.321,14.666,6.368,0.454
2,Benign,Small,185,10.787,11.907,5.153,0.384
3,Malignant,Large,167,18.337,22.178,8.488,0.979
4,Malignant,Medium,39,13.700,16.497,6.446,1.010
5,Malignant,Small,6,11.475,14.152,5.324,1.123


,patient_id,diagnosis_label,tumor_size_grade,mean_radius,area_perimeter_ratio,nuclear_malformation_score
0,PT-1000,Malignant,Large,17.99,8.151466,1.60425
1,PT-1001,Malignant,Large,20.57,9.977427,0.61420
2,PT-1002,Malignant,Large,19.69,9.253846,1.11790
3,PT-1003,Malignant,Small,11.42,4.976798,1.57135
4,PT-1004,Malignant,Large,20.29,9.600296,0.76750
5,PT-1005,Malignant,Medium,12.45,5.778128,1.23450
6,PT-1006,Malignant,Large,18.25,8.695652,0.82920
7,PT-1007,Malignant,Medium,13.71,6.406874,0.79160
